# Module 3: Transactional Orders Fact Table Cleaning & Financial Reconciliation
**Project:** Multi-Source Retail Analytics & Anomaly Detection Pipeline  
**Author:** Piyush Kumar Singh  
**Stack:** Python 3.12+, Pandas, NumPy  
**Input:** `orders_messy.csv` (~50,250 raw order transactions)  
**Output:** `orders_cleaned.csv` (Sanitized Order Fact Table)

---

### Executive Summary & Business Objective
The transactional order fact table (`orders_messy.csv`) forms the analytical core of this retail pipeline, recording 50,000+ line-item purchases across multi-channel POS and digital storefronts. In real-world enterprise environments, order logs collected from heterogeneous POS terminals and mobile apps suffer from widespread data corruption:
1. **Mixed Date Encodings:** Inconsistent date strings (e.g. `YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`) requiring mixed-format datetime coercion.
2. **Polluted Currency Formats:** Numeric monetary fields (`unit_price`, `total_amount`) polluted with currency symbols (`$`, `₹`), thousands separators (`,`), and non-numeric characters.
3. **Mixed Discount Representations:** Discounts stored as heterogeneous percentage strings (e.g. `'15%'`, `'0%'`) rather than numeric decimal ratios.
4. **Invalid Numeric Outliers:** Negative order quantities (`quantity < 0`) caused by inventory write-offs or unhandled return tags, and unrealistic customer ages (`age > 100`).
5. **Categorical Spelling Discrepancies:** Divergent spellings for identical entities (e.g., `'Bengaluru'` vs `'Bangalore'`, `'Canceled'` vs `'Cancelled'`).
6. **Referential Integrity Gaps:** Order transactions missing primary join keys (`customer_id`, `product_id`).
7. **Missing Unit Prices:** Missing pricing records requiring context-aware imputation via product-level group medians.
8. **Discrepant Line-Item Financials:** Null or miscalculated `total_amount` fields requiring mathematical reconstruction ($Quantity 	imes UnitPrice$).

This module implements an end-to-end data hygiene pipeline to produce an audit-ready transactional fact table (`orders_cleaned.csv`).

## 1. Environment Setup & Library Initialization
Import `pandas` and `numpy` for data manipulation, regular expression parsing, and numerical array handling.

In [175]:
import pandas as pd 
import numpy as np

## 2. Ingesting Raw Orders & Initial Profiling Diagnostics
Load `orders_messy.csv` and inspect raw dimensional structure, random samples, data types, and null value counts across all features.

In [176]:
df=pd.read_csv('orders_messy.csv')

In [177]:
df.head()

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,total_amount,payment_method,order_status,sales_channel,shipping_city,customer_age,returned,rating,coupon_code,device,marketing_source
0,ORD0000001,2025-12-11,CUST12577,PROD0232,2,NaN,5%,71452.04,Wallet,Delivered,Website,Mumbai,50.0,0,5.0,NONE,Android,Referral
1,ORD0000002,2026-06-04,CUST15421,PROD0317,3,"$26,541.85",0.1,71662.99,Net Banking,Cancelled,In-store,Chennai,28.0,0,5.0,NONE,Android,Search Ads
2,ORD0000003,2024-12-25,CUST14130,PROD0124,4,11645.74,0.2,37266.37,Wallet,Shipped,Marketplace,Hyderabad,34.0,0,NaN,WELCOME10,Android,Organic
3,ORD0000004,2023-02-04,CUST14117,PROD0012,4,45006.44,0.0,180025.77,Credit Card,Delivered,Marketplace,RANCHI,25.0,0,5.0,NONE,Windows,Email
4,ORD0000005,2023-01-16,CUST13087,PROD0274,2,"₹43,124.43",0.25,NaN,Credit Card,Delivered,In-store,Chennai,33.0,0,2.0,NONE,Other,Social Media


In [178]:
df.sample(10)

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,total_amount,payment_method,order_status,sales_channel,shipping_city,customer_age,returned,rating,coupon_code,device,marketing_source
47967,ORD0047968,2022-04-09,CUST12343,PROD0314,1,34788.78,5%,33049.34,Cash,Delivered,Mobile App,Kolkata,27.0,0,NaN,NONE,Android,Social Media
9435,ORD0009436,2023-02-13,CUST15380,PROD0100,2,"$25,614.40",0.15,"43,544.47 USD",Credit Card,Delivered,Mobile App,Jaipur,43.0,0,5.0,NONE,Windows,Search Ads
45200,ORD0045201,2022-11-15,CUST15941,PROD0042,4,32577.51,0.2,104248.03,UPI,Delivered,Marketplace,Jaipur,60.0,0,NaN,NONE,Other,Social Media
5,ORD0000006,2026-04-12,CUST10916,PROD0217,2,33498.97,0.0,66997.94,Wallet,Delivered,Website,Ranchi,35.0,0,3.0,NaN,Other,Social Media
25154,ORD0025155,2022-04-03,CUST15149,PROD0205,2,6387.25,0.15,10858.33,Credit Card,Delivered,Marketplace,Delhi,45.0,0,NaN,NONE,iOS,Social Media
4214,ORD0004215,2025-03-01,CUST13581,PROD0011,3,8992.43,0.05,25628.43,Credit Card,Delivered,Website,Lucknow,43.0,0,5.0,NONE,Mac,Referral
6982,ORD0006983,2024-09-19,CUST12134,PROD0010,2,4181.86,0%,8363.71,Net Banking,Delivered,Website,Kolkata,60.0,0,NaN,NaN,Mac,Organic
4874,ORD0004875,2024-08-19,CUST10590,PROD0081,1,34826.88,0%,34826.88,NaN,Cancelled,In-store,Bengaluru,41.0,0,5.0,NONE,Android,Search Ads
32276,ORD0032277,2024-02-10,CUST12672,PROD0152,5,"$22,187.00",0.0,NaN,Cash,CANCELED,Marketplace,Lucknow,74.0,0,4.0,NaN,Windows,Email
28446,ORD0028447,2025-07-12,CUST13808,PROD0305,5,8448.54,0.05,40130.57,Net Banking,Delivered,In-store,Ranchi,72.0,0,3.0,NEWUSER,Windows,Organic


In [179]:
df.shape

(50250, 18)

In [180]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50250 entries, 0 to 50249
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          50250 non-null  str    
 1   order_date        49848 non-null  str    
 2   customer_id       50049 non-null  str    
 3   product_id        50100 non-null  str    
 4   quantity          50250 non-null  int64  
 5   unit_price        48858 non-null  str    
 6   discount          50250 non-null  str    
 7   total_amount      49108 non-null  str    
 8   payment_method    49497 non-null  str    
 9   order_status      50250 non-null  str    
 10  sales_channel     50250 non-null  str    
 11  shipping_city     48744 non-null  str    
 12  customer_age      48242 non-null  float64
 13  returned          50250 non-null  int64  
 14  rating            40752 non-null  float64
 15  coupon_code       41171 non-null  str    
 16  device            50250 non-null  str    
 17  mark

In [181]:
df.dtypes

order_id                str
order_date              str
customer_id             str
product_id              str
quantity              int64
unit_price              str
discount                str
total_amount            str
payment_method          str
order_status            str
sales_channel           str
shipping_city           str
customer_age        float64
returned              int64
rating              float64
coupon_code             str
device                  str
marketing_source        str
dtype: object

In [182]:
df.isnull().sum()

order_id               0
order_date           402
customer_id          201
product_id           150
quantity               0
unit_price          1392
discount               0
total_amount        1142
payment_method       753
order_status           0
sales_channel          0
shipping_city       1506
customer_age        2008
returned               0
rating              9498
coupon_code         9079
device                 0
marketing_source       0
dtype: int64

In [183]:
df.duplicated().sum()

np.int64(250)

In [184]:
df[df['customer_id'].duplicated()]

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,total_amount,payment_method,order_status,sales_channel,shipping_city,customer_age,returned,rating,coupon_code,device,marketing_source
15,ORD0000016,2022-01-17,CUST14438,PROD0323,5,10689.1,0%,53445.49,Wallet,Delivered,In-store,Pune,73.0,1,4.0,WELCOME10,Windows,Social Media
122,ORD0000123,2025-07-21,NaN,PROD0309,1,"$31,562.16",0.2,25249.73,Wallet,Delivered,In-store,Hyderabad,62.0,0,3.0,NONE,Android,Email
161,ORD0000162,2024-07-06,NaN,PROD0057,2,19506.17,0.0,39012.35,Net Banking,Shipped,Website,Ranchi,46.0,1,3.0,NONE,Windows,Social Media
278,ORD0000279,2024-02-07,CUST15808,PROD0262,1,"₹9,644.47",0.15,8197.8,Credit Card,Cancelled,Marketplace,Kolkata,69.0,0,3.0,NONE,Windows,Email
291,ORD0000292,03-Feb-2024,CUST10159,PROD0314,1,31277.13,0.1,"₹28,149.42",Cash,Shipped,Mobile App,Bengaluru,40.0,0,2.0,NONE,Android,Email
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50245,ORD0041713,2022-11-07,CUST11065,PROD0170,2,36395.92,5%,69152.25,Cash,Pending,Marketplace,Delhi,39.0,0,4.0,NaN,Other,Social Media
50246,ORD0011094,2023-04-30,CUST12822,PROD0004,2,32321.89,0.25,48482.84,Wallet,Pending,Mobile App,Hyderabad,35.0,0,4.0,FESTIVE20,Mac,Referral
50247,ORD0008696,2024-11-30,CUST12655,PROD0208,1,"$5,956.92",0.1,5361.23,Credit Card,Delivered,Mobile App,Mumbai,32.0,0,NaN,WELCOME10,Mac,Search Ads
50248,ORD0007997,2022-10-30,CUST11802,PROD0099,1,"₹39,839.61",0.1,35855.65,Credit Card,Delivered,Mobile App,Pune,18.0,0,NaN,NaN,Android,Social Media


## 3. Mixed-Format Datetime Parsing & Missing Date Imputation
Order dates arrive in varied formats across regional POS systems. We leverage `pd.to_datetime(format='mixed', errors='coerce')` to parse disparate date strings into uniform timestamps, then apply forward-fill (`ffill()`) to handle sporadic timestamp dropouts.

In [185]:
df['order_date']=pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

In [ ]:
df.dtypes

order_id                       str
order_date          datetime64[us]
customer_id                    str
product_id                     str
quantity                     int64
unit_price                     str
discount                       str
total_amount                   str
payment_method                 str
order_status                   str
sales_channel                  str
shipping_city                  str
customer_age               float64
returned                     int64
rating                     float64
coupon_code                    str
device                         str
marketing_source               str
dtype: object

In [187]:
df['order_date'].isnull().sum()

np.int64(402)

In [188]:
df['order_date']=df['order_date'].ffill(inplace=True)

C:\Users\piyus\AppData\Local\Temp\ipykernel_25928\1781033208.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['order_date']=df['order_date'].ffill(inplace=True)


In [189]:
df['order_date'].isnull().sum()

np.int64(0)

## 4. Unit Price Currency Sanitization & Type Coercion
The `unit_price` column is stored as an object string contaminated with currency symbols (`$`, `₹`), commas, and whitespace. We strip these non-numeric artifacts to restore pure numerical floats.

In [190]:
df.columns

Index(['order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
       'unit_price', 'discount', 'total_amount', 'payment_method',
       'order_status', 'sales_channel', 'shipping_city', 'customer_age',
       'returned', 'rating', 'coupon_code', 'device', 'marketing_source'],
      dtype='str')

In [191]:
df['total_amount'].unique()

<StringArray>
[ '71452.04',  '71662.99',  '37266.37', '180025.77',         nan,  '66997.94',
  '20196.22',  '23998.01',  '88437.87', '106588.44',
 ...
 '172768.31',  '112707.6', '165923.46', '160217.86',  '39504.63',  '80099.79',
  '27115.26', '100392.03',  '90719.45', '154671.15']
Length: 48600, dtype: str

In [192]:
df['total_amount'].sample(20)

7760        38529.86
43995       42225.71
21467       44662.24
24062       76519.18
16059       64938.17
4581         5907.61
4853        44759.08
31762       189956.1
42393        1913.19
48081       39118.12
39037       35111.53
36472      108356.52
28281       40689.61
47215       85819.99
3024        15263.93
5791     ₹116,872.32
38306      22360.09 
30928       17127.85
9594      ₹68,513.91
30999      121050.22
Name: total_amount, dtype: str

In [193]:
df['unit_price'].sample(5)

48451    11797.91
20181     3965.73
22205    18262.05
29637    12033.99
16616     4793.38
Name: unit_price, dtype: str

In [194]:
df['unit_price'].value_counts()

unit_price
?           75
—           69
unknown     62
6431.23      3
28010.61     2
            ..
40049.9      1
27115.26     1
25098.01     1
30239.82     1
45491.51     1
Name: count, Length: 48224, dtype: int64

In [195]:
df['unit_price']=df['unit_price'].str.replace('$','',regex=False)
df['unit_price']=df['unit_price'].str.replace('₹','',regex=False)
df['unit_price']=df['unit_price'].str.replace('USD','',regex=False)
df['unit_price']=df['unit_price'].str.replace(',','',regex=False)
df['unit_price']=df['unit_price'].str.replace('?','',regex=False)
df['unit_price']=df['unit_price'].str.replace('unknown','',regex=False)

df['unit_price']=pd.to_numeric(df['unit_price'], errors='coerce')


In [196]:
df['unit_price'].sample(10)

14530    39581.08
49882    31650.78
29565    34466.33
33153    37686.24
14415     2334.04
33380    22523.78
42791    25975.05
5067     38464.85
31948    41969.13
37007    29129.49
Name: unit_price, dtype: float64

## 5. Discount Format Normalization: String to Decimal Conversion
The `discount` field contains mixed percentage strings (e.g. `'15%'`, `'20%'`). We map these percentage representations into true mathematical decimal fractions ($0.15, 0.20$) and coerce them to numeric types.

In [197]:
df.columns

Index(['order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
       'unit_price', 'discount', 'total_amount', 'payment_method',
       'order_status', 'sales_channel', 'shipping_city', 'customer_age',
       'returned', 'rating', 'coupon_code', 'device', 'marketing_source'],
      dtype='str')

In [198]:
df['discount'].unique()

<StringArray>
[  '5%',  '0.1',  '0.2',  '0.0', '0.25',  '0.3', '0.15', '0.05',  '15%',
  '10%',   '0%',  '20%',  '25%',  '30%']
Length: 14, dtype: str

In [199]:
print(df['discount'].dtypes)

str


In [200]:
discount_map=({
    '15%':'0.15',
    '10%':'0.1',   
    '0%':'0.0', 
    '5%':'0.05' ,
    '20%':'0.20',  
    '25%':'0.25',  
    '30%':'0.30'
})

df['discount']=df['discount'].replace(discount_map, inplace=True)

C:\Users\piyus\AppData\Local\Temp\ipykernel_25928\81722443.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['discount']=df['discount'].replace(discount_map, inplace=True)


In [201]:
df['discount'].unique()

<StringArray>
['0.05', '0.1', '0.2', '0.0', '0.25', '0.3', '0.15', '0.20', '0.30']
Length: 9, dtype: str

In [202]:
df['discount']=pd.to_numeric(df['discount'], errors='coerce')

In [203]:
print(df['discount'].dtypes)

float64


In [204]:
df.dtypes

order_id                       str
order_date          datetime64[us]
customer_id                    str
product_id                     str
quantity                     int64
unit_price                 float64
discount                   float64
total_amount                   str
payment_method                 str
order_status                   str
sales_channel                  str
shipping_city                  str
customer_age               float64
returned                     int64
rating                     float64
coupon_code                    str
device                         str
marketing_source               str
dtype: object

## 6. Categorical Normalization: Payment Methods, Order Status & City Entities
Standardize categorical text across three business dimensions:
1. **Payment Methods:** Strip whitespace and convert to uniform UPPERCASE (e.g. `'CREDIT CARD'`, `'UPI'`).
2. **Order Status:** Standardize to Title Case and resolve regional spelling variants (e.g. mapping `'Canceled'` to `'Cancelled'`).
3. **Shipping City:** Normalize city names by stripping whitespace, title-casing, and consolidating entity variants (e.g. mapping `'Bengaluru'` to `'Bangalore'`).

In [205]:
df['payment_method'].unique()

<StringArray>
[       'Wallet',   'Net Banking',   'Credit Card',        'wallet',
           'UPI',   'net banking',          'Cash',    'Debit Card',
             nan,    'Debit card',           'upi', ' CREDIT CARD ',
    'debit card',   'Credit card',      ' WALLET ',         ' UPI ',
        ' CASH ',           'Upi',    'NetBanking',          'cash',
   'credit card']
Length: 21, dtype: str

In [206]:
df['payment_method']=df['payment_method'].str.strip().str.upper()

In [207]:
df['payment_method'].unique()

<StringArray>
[     'WALLET', 'NET BANKING', 'CREDIT CARD',         'UPI',        'CASH',
  'DEBIT CARD',           nan,  'NETBANKING']
Length: 8, dtype: str

In [208]:
df['payment_method'].isnull().sum()

np.int64(753)

In [209]:
df['order_status'].unique()

<StringArray>
[  'Delivered',   'Cancelled',     'Shipped',     'Pending',    'Returned',
   'DELIVERED',     'shipped',   'delivered',   'cancelled', ' Delivered ',
    'returned',     'SHIPPED',     'PENDING',     'pending',    'RETURNED',
    'CANCELED']
Length: 16, dtype: str

In [210]:
df['order_status']=df['order_status'].str.strip().str.title()

In [211]:
df['order_status']=df['order_status'].replace('Canceled','Cancelled')

In [212]:
df['order_status'].unique()

<StringArray>
['Delivered', 'Cancelled', 'Shipped', 'Pending', 'Returned']
Length: 5, dtype: str

In [213]:
df['shipping_city']=df['shipping_city'].str.strip().str.title()

In [214]:
df['shipping_city']=df['shipping_city'].replace('Bengaluru','Bangalore')

In [215]:
df['shipping_city'].unique()

<StringArray>
[   'Mumbai',   'Chennai', 'Hyderabad',    'Ranchi',      'Pune',   'Lucknow',
    'Jaipur', 'Bangalore',     'Delhi',   'Kolkata',         nan]
Length: 11, dtype: str

## 7. Quantity Outlier Treatment & Missing Categorical Mode Imputation
- Transactions with `quantity < 0` reflect inventory reversals or malformed records; replace these negative values with `np.nan` for subsequent median imputation.
- Impute missing `shipping_city` and `payment_method` observations using their respective categorical modes (`.mode()[0]`).

In [216]:
df.columns

Index(['order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
       'unit_price', 'discount', 'total_amount', 'payment_method',
       'order_status', 'sales_channel', 'shipping_city', 'customer_age',
       'returned', 'rating', 'coupon_code', 'device', 'marketing_source'],
      dtype='str')

In [217]:
df['marketing_source'].unique()

<StringArray>
['Referral', 'Search Ads', 'Organic', 'Email', 'Social Media']
Length: 5, dtype: str

In [218]:
df.describe()

,order_date,quantity,unit_price,discount,customer_age,returned,rating
count,50250,50250.000000,48652.000000,50250.000000,48242.000000,50250.000000,40752.000000
mean,2024-04-30 03:47:42.376119,2.986249,24067.824294,0.111395,46.258178,0.069970,3.903514
min,2022-01-01 00:00:00,-4.000000,720.280000,0.000000,5.000000,0.000000,0.000000
25%,2023-03-04 06:00:00,2.000000,11812.932500,0.000000,32.000000,0.000000,3.000000
50%,2024-05-02 00:00:00,3.000000,24090.185000,0.100000,46.000000,0.000000,4.000000
75%,2025-06-26 00:00:00,4.000000,35076.052500,0.150000,60.000000,0.000000,5.000000
max,2026-12-07 00:00:00,5.000000,57395.900000,0.300000,150.000000,1.000000,10.000000
std,NaN,1.426079,13891.655587,0.086056,16.599380,0.255099,1.148926


In [219]:
# df.loc[(df['quantity']<0),'quantity']=np.nan

In [220]:
df.loc[(df['quantity']<0),'quantity']=np.nan

In [221]:
df.duplicated().sum()

np.int64(248)

In [222]:
df.isnull().sum()

order_id               0
order_date             0
customer_id          201
product_id           150
quantity              60
unit_price          1598
discount               0
total_amount        1142
payment_method       753
order_status           0
sales_channel          0
shipping_city       1506
customer_age        2008
returned               0
rating              9498
coupon_code         9079
device                 0
marketing_source       0
dtype: int64

In [223]:
df.describe()

,order_date,quantity,unit_price,discount,customer_age,returned,rating
count,50250,50190.000000,48652.000000,50250.000000,48242.000000,50250.000000,40752.000000
mean,2024-04-30 03:47:42.376119,2.992887,24067.824294,0.111395,46.258178,0.069970,3.903514
min,2022-01-01 00:00:00,1.000000,720.280000,0.000000,5.000000,0.000000,0.000000
25%,2023-03-04 06:00:00,2.000000,11812.932500,0.000000,32.000000,0.000000,3.000000
50%,2024-05-02 00:00:00,3.000000,24090.185000,0.100000,46.000000,0.000000,4.000000
75%,2025-06-26 00:00:00,4.000000,35076.052500,0.150000,60.000000,0.000000,5.000000
max,2026-12-07 00:00:00,5.000000,57395.900000,0.300000,150.000000,1.000000,10.000000
std,NaN,1.413414,13891.655587,0.086056,16.599380,0.255099,1.148926


In [224]:
df['shipping_city']=df['shipping_city'].fillna(df['shipping_city'].mode()[0])

In [225]:
df['payment_method']=df['payment_method'].fillna(df['payment_method'].mode()[0])

## 8. Customer Age & Behavioral Feature Cleaning
- Flag unrealistic customer ages (`age > 100`) as `np.nan` and impute using the population median.
- Fill missing `coupon_code` values with the explicit placeholder `'NONE'`.
- Impute missing product `rating` entries using the median customer satisfaction score.

In [226]:
df['customer_age'].describe()

count    48242.000000
mean        46.258178
std         16.599380
min          5.000000
25%         32.000000
50%         46.000000
75%         60.000000
max        150.000000
Name: customer_age, dtype: float64

In [227]:
df.loc[df['customer_age'] > 100, 'customer_age'] = np.nan

In [228]:
df['customer_age'].unique()

array([50., 28., 34., 25., 33., 35., 32., 73., 63., 66., 67., 18., 39.,
       48., 60., 44., 46., 37., 51., 30., 47., 26., 49., 42., 40.,  5.,
       54., 57., 43., 58., 55., 19., 23., nan, 36., 71., 21., 56., 52.,
       65., 53., 41., 61., 69., 72., 20., 62., 59., 27., 70., 74., 64.,
       31., 29., 45., 38., 24., 68., 22.,  8.])

In [229]:
df['customer_age']=df['customer_age'].fillna(df['customer_age'].median())

In [230]:
df.isna().sum()

order_id               0
order_date             0
customer_id          201
product_id           150
quantity              60
unit_price          1598
discount               0
total_amount        1142
payment_method         0
order_status           0
sales_channel          0
shipping_city          0
customer_age           0
returned               0
rating              9498
coupon_code         9079
device                 0
marketing_source       0
dtype: int64

In [231]:
df.describe()

,order_date,quantity,unit_price,discount,customer_age,returned,rating
count,50250,50190.000000,48652.000000,50250.000000,50250.000000,50250.000000,40752.000000
mean,2024-04-30 03:47:42.376119,2.992887,24067.824294,0.111395,46.198826,0.069970,3.903514
min,2022-01-01 00:00:00,1.000000,720.280000,0.000000,5.000000,0.000000,0.000000
25%,2023-03-04 06:00:00,2.000000,11812.932500,0.000000,33.000000,0.000000,3.000000
50%,2024-05-02 00:00:00,3.000000,24090.185000,0.100000,46.000000,0.000000,4.000000
75%,2025-06-26 00:00:00,4.000000,35076.052500,0.150000,60.000000,0.000000,5.000000
max,2026-12-07 00:00:00,5.000000,57395.900000,0.300000,74.000000,1.000000,10.000000
std,NaN,1.413414,13891.655587,0.086056,16.118523,0.255099,1.148926


In [232]:
df['coupon_code']=df['coupon_code'].fillna('NONE')

In [233]:
df['coupon_code'].unique()

<StringArray>
['NONE', 'WELCOME10', 'NEWUSER', 'FESTIVE20']
Length: 4, dtype: str

In [234]:
df['rating'].unique()

array([ 5., nan,  2.,  3.,  4.,  1., 10.,  6.,  0.])

In [235]:
df['rating']=df['rating'].fillna(df['rating'].median())

In [236]:
df['rating'].isnull().sum()

np.int64(0)

In [237]:
df.describe()

,order_date,quantity,unit_price,discount,customer_age,returned,rating
count,50250,50190.000000,48652.000000,50250.000000,50250.000000,50250.000000,50250.000000
mean,2024-04-30 03:47:42.376119,2.992887,24067.824294,0.111395,46.198826,0.069970,3.921751
min,2022-01-01 00:00:00,1.000000,720.280000,0.000000,5.000000,0.000000,0.000000
25%,2023-03-04 06:00:00,2.000000,11812.932500,0.000000,33.000000,0.000000,4.000000
50%,2024-05-02 00:00:00,3.000000,24090.185000,0.100000,46.000000,0.000000,4.000000
75%,2025-06-26 00:00:00,4.000000,35076.052500,0.150000,60.000000,0.000000,5.000000
max,2026-12-07 00:00:00,5.000000,57395.900000,0.300000,74.000000,1.000000,10.000000
std,NaN,1.413414,13891.655587,0.086056,16.118523,0.255099,1.035349


In [238]:
df.isnull().sum()


order_id               0
order_date             0
customer_id          201
product_id           150
quantity              60
unit_price          1598
discount               0
total_amount        1142
payment_method         0
order_status           0
sales_channel          0
shipping_city          0
customer_age           0
returned               0
rating                 0
coupon_code            0
device                 0
marketing_source       0
dtype: int64

## 9. Referential Integrity Filtering
Transactions lacking a valid `customer_id` or `product_id` cannot be joined to dimension tables in a Star-Schema data warehouse. Drop rows with null foreign keys to guarantee relational integrity.

In [239]:
df.sample(10)

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,total_amount,payment_method,order_status,sales_channel,shipping_city,customer_age,returned,rating,coupon_code,device,marketing_source
22512,ORD0022513,2023-01-21,CUST14150,PROD0151,4.0,11616.91,0.10,41820.88,DEBIT CARD,Delivered,In-store,Ranchi,23.0,0,5.0,NONE,Mac,Organic
13705,ORD0013706,2024-08-10,CUST11952,PROD0069,4.0,15499.69,0.25,"46,499.06 USD",DEBIT CARD,Delivered,Mobile App,Jaipur,67.0,0,5.0,NONE,iOS,Social Media
46351,ORD0046352,2023-12-09,CUST12434,PROD0288,4.0,41587.81,0.15,141398.57,UPI,Delivered,Marketplace,Delhi,52.0,0,4.0,FESTIVE20,Android,Email
42444,ORD0042445,2024-02-19,CUST11979,PROD0258,1.0,46043.91,0.15,39137.33,CREDIT CARD,Delivered,Marketplace,Pune,30.0,0,4.0,WELCOME10,Android,Organic
27543,ORD0027544,2023-11-27,CUST12976,PROD0238,2.0,4310.99,0.10,7759.79,DEBIT CARD,Cancelled,Website,Ranchi,23.0,0,3.0,WELCOME10,Other,Search Ads
11851,ORD0011852,2025-10-04,CUST11509,PROD0311,4.0,17805.93,0.15,"60,540.16 USD",CREDIT CARD,Delivered,Website,Kolkata,70.0,0,5.0,FESTIVE20,Android,Organic
34902,ORD0034903,2022-07-06,CUST11570,PROD0254,3.0,14542.41,0.10,39264.51,NET BANKING,Cancelled,In-store,Hyderabad,46.0,0,4.0,WELCOME10,Mac,Email
49059,ORD0049060,2025-07-01,CUST12010,PROD0138,3.0,28005.31,0.15,71413.55,DEBIT CARD,Delivered,Marketplace,Ranchi,65.0,0,4.0,NEWUSER,iOS,Referral
24050,ORD0024051,2025-05-18,CUST12680,PROD0325,5.0,15909.68,0.00,79548.39,WALLET,Delivered,Mobile App,Bangalore,67.0,0,5.0,NONE,iOS,Referral
45166,ORD0045167,2024-10-09,CUST12330,PROD0228,5.0,37457.46,0.00,NaN,DEBIT CARD,Delivered,Mobile App,Bangalore,23.0,0,5.0,NONE,Other,Search Ads


In [240]:
df=df.dropna(subset='customer_id')

In [241]:
df.shape

(50049, 18)

In [242]:
df.isnull().sum()

order_id               0
order_date             0
customer_id            0
product_id           150
quantity              59
unit_price          1591
discount               0
total_amount        1137
payment_method         0
order_status           0
sales_channel          0
shipping_city          0
customer_age           0
returned               0
rating                 0
coupon_code            0
device                 0
marketing_source       0
dtype: int64

In [243]:
df.dropna(subset='product_id',inplace=True)

In [244]:
df.isnull().sum()

order_id               0
order_date             0
customer_id            0
product_id             0
quantity              59
unit_price          1586
discount               0
total_amount        1134
payment_method         0
order_status           0
sales_channel          0
shipping_city          0
customer_age           0
returned               0
rating                 0
coupon_code            0
device                 0
marketing_source       0
dtype: int64

## 10. Advanced Product-Level Group Median Imputation for Unit Price
When `unit_price` is missing, global median imputation would distort catalog pricing tiers. Instead, we use an advanced grouped transform:
$$\text{Imputed Price}_i = \text{Median}(\text{UnitPrice}_{\text{Product}(i)})$$
This imputes missing prices using the specific median price of that exact SKU.

In [245]:
df['quantity']=df['quantity'].fillna(df['quantity'].median())

In [246]:
df[df['unit_price'].isnull()]

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,total_amount,payment_method,order_status,sales_channel,shipping_city,customer_age,returned,rating,coupon_code,device,marketing_source
0,ORD0000001,2025-12-11,CUST12577,PROD0232,2.0,NaN,0.05,71452.04,WALLET,Delivered,Website,Mumbai,50.0,0,5.0,NONE,Android,Referral
21,ORD0000022,2026-04-16,CUST11247,PROD0033,2.0,NaN,0.20,20526.62,NET BANKING,Pending,In-store,Jaipur,51.0,0,4.0,NONE,iOS,Email
50,ORD0000051,2025-08-05,CUST12180,PROD0026,4.0,NaN,0.30,79774.62,CREDIT CARD,Delivered,Mobile App,Delhi,49.0,0,5.0,WELCOME10,Mac,Email
55,ORD0000056,2023-03-27,CUST14524,PROD0099,4.0,NaN,0.00,170720.97,UPI,Delivered,Mobile App,Chennai,46.0,0,5.0,NONE,Mac,Referral
64,ORD0000065,2025-08-23,CUST11136,PROD0192,4.0,NaN,0.05,62626.16,CREDIT CARD,Delivered,Website,Hyderabad,39.0,0,5.0,NONE,Other,Social Media
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50043,ORD0019368,2023-07-15,CUST13370,PROD0080,4.0,NaN,0.00,19390.14,DEBIT CARD,Returned,In-store,Ranchi,73.0,0,4.0,NONE,Windows,Email
50091,ORD0015222,2022-12-29,CUST14313,PROD0023,3.0,NaN,0.10,90567.52,NETBANKING,Pending,Mobile App,Pune,60.0,0,4.0,NONE,Windows,Organic
50128,ORD0004004,2025-12-06,CUST15028,PROD0151,1.0,NaN,0.25,9166.48,DEBIT CARD,Delivered,Marketplace,Chennai,74.0,0,3.0,NONE,iOS,Social Media
50200,ORD0045100,2025-07-17,CUST11232,PROD0257,1.0,NaN,0.20,26282.95,UPI,Delivered,In-store,Delhi,24.0,0,5.0,NONE,Other,Social Media


In [247]:
df['unit_price']=df['unit_price'].fillna(df.groupby('product_id')['unit_price'].transform('median'))

In [248]:
df['unit_price'].isnull().sum()

np.int64(0)

## 11. Total Revenue Financial Reconciliation & String Cleanup
Clean any lingering currency artifacts (`$`, `,`) in `total_amount`, and reconcile missing line-item totals using the deterministic identity:
$$\text{TotalAmount} = \text{Quantity} \times \text{UnitPrice}$$

In [249]:
df['total_amount']=df['total_amount'].fillna(df['quantity']*df['unit_price'])

In [250]:
df.isnull().sum()

order_id            0
order_date          0
customer_id         0
product_id          0
quantity            0
unit_price          0
discount            0
total_amount        0
payment_method      0
order_status        0
sales_channel       0
shipping_city       0
customer_age        0
returned            0
rating              0
coupon_code         0
device              0
marketing_source    0
dtype: int64

In [251]:
df.to_csv('orders_clean.csv', index=False)

In [252]:
df.dtypes

order_id                       str
order_date          datetime64[us]
customer_id                    str
product_id                     str
quantity                   float64
unit_price                 float64
discount                   float64
total_amount                object
payment_method                 str
order_status                   str
sales_channel                  str
shipping_city                  str
customer_age               float64
returned                     int64
rating                     float64
coupon_code                    str
device                         str
marketing_source               str
dtype: object

In [253]:
df['total_amount'].sample(10)

14845       20569.25
49329      102432.14
32843       91604.26
3906          6034.7
14756      115862.41
2727       110223.31
38259       10830.94
39812      150857.12
6616     ₹153,456.09
42977      173230.63
Name: total_amount, dtype: object

In [254]:
df['total_amount']=df['total_amount'].replace('$','',regex=False)
df['total_amount']=df['total_amount'].replace(',','',regex=False)
df['total_amount']=df['total_amount'].replace('₹','',regex=False)
df['total_amount']=df['total_amount'].replace('USD','',regex=False)
df['total_amount']=df['total_amount'].replace('?','',regex=False)

df['total_amount']=pd.to_numeric(df['total_amount'], errors='coerce')

In [255]:
df.isnull().sum()

order_id               0
order_date             0
customer_id            0
product_id             0
quantity               0
unit_price             0
discount               0
total_amount        6698
payment_method         0
order_status           0
sales_channel          0
shipping_city          0
customer_age           0
returned               0
rating                 0
coupon_code            0
device                 0
marketing_source       0
dtype: int64

In [256]:
df['total_amount']=df['total_amount'].fillna(df['quantity']*df['unit_price'])

## 12. Deduplication & Clean Orders Production Export
Audit the sanitized fact table for redundant duplicate rows, drop exact duplicate entries, and export the final dataset to `orders_cleaned.csv`.

In [259]:
df.duplicated().sum()

np.int64(0)

In [258]:
df=df.drop_duplicates()

In [260]:
df.to_csv('orders_cleaned.csv',index=False)